# Business Analytics Project — Retail Sales, Customer & Profit Analytics

**Dataset:** `synthetic_business_sales_dataset.csv`  
**Source project document:** Business Analytics Project – Retail Sales, Customer and Profit Analytics  
**Random seed:** 42

This notebook reproduces the analytical workflow described in the project document using the supplied 1,200-row, 18-field synthetic retail dataset.

### Analyses included
EDA, descriptive statistics, distribution analysis, correlation, regression, dashboard-ready summaries, probability, decision analysis, sampling, confidence intervals, bootstrapping, hypothesis testing, simulated A/B testing, time-series forecasting, forecast accuracy, optimization, simulation, ANOVA, experimental design and factorial experiment.


In [ ]:
# 1. Imports and display settings
import os
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from scipy import stats
import statsmodels.api as sm
from statsmodels.tsa.holtwinters import ExponentialSmoothing
from statsmodels.tsa.arima.model import ARIMA

np.random.seed(42)
pd.set_option("display.max_columns", None)
pd.set_option("display.float_format", lambda x: f"{x:,.2f}")

print("Libraries loaded.")


In [ ]:
# 2. Load the supplied CSV
# The first path is for the uploaded file in this environment.
# If running in Google Colab, upload the CSV when prompted if it is not found.

csv_candidates = [
    "/mnt/data/synthetic_business_sales_dataset.csv",
    "synthetic_business_sales_dataset.csv"
]

csv_path = next((p for p in csv_candidates if os.path.exists(p)), None)

if csv_path is None:
    try:
        from google.colab import files
        uploaded = files.upload()
        csv_path = next(iter(uploaded))
    except Exception:
        raise FileNotFoundError(
            "CSV not found. Place synthetic_business_sales_dataset.csv in the notebook folder."
        )

df = pd.read_csv(csv_path)
df["Order_Date"] = pd.to_datetime(df["Order_Date"])

print(f"Loaded: {csv_path}")
print(f"Rows: {df.shape[0]}, Columns: {df.shape[1]}")
display(df.head())


## 3. Dataset Description

In [ ]:
# Dataset structure and basic validation
display(pd.DataFrame({
    "Column": df.columns,
    "Data Type": df.dtypes.astype(str),
    "Missing Values": df.isna().sum().values,
    "Unique Values": [df[c].nunique() for c in df.columns]
}))

print("Duplicate rows:", df.duplicated().sum())
print("Date range:", df["Order_Date"].min().date(), "to", df["Order_Date"].max().date())


## 4. Data Preprocessing

In [ ]:
# Validate numeric fields and categorical fields
numeric_cols = [
    "Age", "Quantity", "Unit_Price", "Discount_Pct",
    "Sales", "Profit", "Rating", "Delivery_Days", "Returned"
]

categorical_cols = [
    "Order_ID", "Customer_ID", "Region", "Customer_Segment",
    "Channel", "Payment_Method", "Category", "Product"
]

print("Missing values:", int(df.isna().sum().sum()))
print("Duplicate rows:", int(df.duplicated().sum()))
print("Negative Sales:", int((df["Sales"] < 0).sum()))
print("Non-positive Quantity:", int((df["Quantity"] <= 0).sum()))
print("Invalid Rating:", int(((df["Rating"] < 0) | (df["Rating"] > 5)).sum()))

# Useful derived fields
df["Month"] = df["Order_Date"].dt.to_period("M").astype(str)
df["Year"] = df["Order_Date"].dt.year
df["Month_Num"] = df["Order_Date"].dt.month
df["AOV"] = df["Sales"]  # each row represents one transaction/order


## 5. Exploratory Data Analysis

In [ ]:
# Category, region, channel and segment summaries
for col in ["Category", "Region", "Channel", "Customer_Segment"]:
    print(f"\n--- {col} ---")
    display(df[col].value_counts().rename("Orders").to_frame())

# Category sales
category_sales = df.groupby("Category").agg(
    Orders=("Order_ID", "count"),
    Sales=("Sales", "sum"),
    Profit=("Profit", "sum"),
    Avg_Rating=("Rating", "mean")
).sort_values("Sales", ascending=False)
display(category_sales)


In [ ]:
# EDA charts
fig, ax = plt.subplots(figsize=(9,5))
df.groupby("Category")["Sales"].sum().sort_values().plot(kind="barh", ax=ax)
ax.set_title("Total Sales by Category")
ax.set_xlabel("Sales (₹)")
plt.tight_layout()
plt.show()

fig, ax = plt.subplots(figsize=(9,5))
df.groupby("Region")["Sales"].sum().sort_values().plot(kind="bar", ax=ax)
ax.set_title("Total Sales by Region")
ax.set_ylabel("Sales (₹)")
plt.xticks(rotation=0)
plt.tight_layout()
plt.show()

fig, ax = plt.subplots(figsize=(10,5))
monthly_sales = df.groupby("Month")["Sales"].sum()
monthly_sales.plot(ax=ax)
ax.set_title("Monthly Sales")
ax.set_xlabel("Month")
ax.set_ylabel("Sales (₹)")
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()


In [ ]:
# Distribution plots
fig, ax = plt.subplots(figsize=(9,5))
ax.hist(df["Sales"], bins=30)
ax.set_title("Distribution of Sales")
ax.set_xlabel("Sales (₹)")
ax.set_ylabel("Frequency")
plt.tight_layout()
plt.show()

fig, ax = plt.subplots(figsize=(9,5))
ax.boxplot(df["Sales"], vert=False)
ax.set_title("Sales Boxplot")
ax.set_xlabel("Sales (₹)")
plt.tight_layout()
plt.show()


## 6. Descriptive Statistical Analysis

In [ ]:
# Mean, median, mode, min, max, variance and standard deviation
desc_vars = ["Quantity", "Unit_Price", "Discount_Pct", "Sales", "Profit", "Rating", "Delivery_Days"]

rows = []
for col in desc_vars:
    s = df[col].dropna()
    rows.append({
        "Variable": col,
        "Mean": s.mean(),
        "Median": s.median(),
        "Mode": s.mode().iloc[0] if not s.mode().empty else np.nan,
        "Min": s.min(),
        "Max": s.max(),
        "Variance": s.var(),
        "Std Dev": s.std()
    })

descriptive_stats = pd.DataFrame(rows).set_index("Variable")
display(descriptive_stats)

print("Total Sales = ₹{:,.2f}".format(df["Sales"].sum()))
print("Total Profit = ₹{:,.2f}".format(df["Profit"].sum()))
print("Average Order Value = ₹{:,.2f}".format(df["Sales"].mean()))
print("Return Rate = {:.2%}".format(df["Returned"].mean()))


## 7. Correlation Analysis

In [ ]:
corr_cols = ["Quantity", "Unit_Price", "Discount_Pct", "Sales", "Profit", "Rating", "Delivery_Days", "Returned"]
corr = df[corr_cols].corr()
display(corr)

fig, ax = plt.subplots(figsize=(9,7))
im = ax.imshow(corr, aspect="auto")
ax.set_xticks(range(len(corr.columns)))
ax.set_yticks(range(len(corr.columns)))
ax.set_xticklabels(corr.columns, rotation=45, ha="right")
ax.set_yticklabels(corr.columns)
ax.set_title("Correlation Matrix")
for i in range(len(corr)):
    for j in range(len(corr)):
        ax.text(j, i, f"{corr.iloc[i,j]:.2f}", ha="center", va="center")
plt.colorbar(im, ax=ax)
plt.tight_layout()
plt.show()


## 8. Regression Analysis

In [ ]:
# Reproduce the project's one-variable regression: Sales ~ Discount_Pct
X = sm.add_constant(df["Discount_Pct"])
y = df["Sales"]
model = sm.OLS(y, X).fit()

print(model.summary())
print("\nRegression equation:")
print(f"Sales = {model.params['const']:,.2f} + ({model.params['Discount_Pct']:.2f} × Discount_Pct)")
print(f"R² = {model.rsquared:.4f}")
print(f"RMSE = ₹{np.sqrt(np.mean(model.resid**2)):,.2f}")

fig, ax = plt.subplots(figsize=(9,5))
ax.scatter(df["Discount_Pct"], df["Sales"], alpha=0.5)
xline = np.linspace(df["Discount_Pct"].min(), df["Discount_Pct"].max(), 100)
ax.plot(xline, model.params["const"] + model.params["Discount_Pct"] * xline)
ax.set_title("Sales vs Discount Percentage")
ax.set_xlabel("Discount (%)")
ax.set_ylabel("Sales (₹)")
plt.tight_layout()
plt.show()


## 9. Business Intelligence Dashboard — KPI Tables

In [ ]:
# Dashboard-ready KPI values and filterable summary tables
kpis = pd.DataFrame({
    "KPI": ["Total Sales", "Total Profit", "Orders", "Average Order Value", "Return Rate", "Average Rating"],
    "Value": [
        df["Sales"].sum(),
        df["Profit"].sum(),
        df["Order_ID"].nunique(),
        df["Sales"].mean(),
        df["Returned"].mean(),
        df["Rating"].mean()
    ]
})
display(kpis)

dashboard_summary = df.groupby(["Region", "Category", "Channel", "Customer_Segment"]).agg(
    Orders=("Order_ID", "count"),
    Sales=("Sales", "sum"),
    Profit=("Profit", "sum"),
    Return_Rate=("Returned", "mean"),
    Avg_Rating=("Rating", "mean")
).reset_index()

display(dashboard_summary.head(20))


## 10. Probability and Probability Distributions

In [ ]:
q75 = df["Sales"].quantile(0.75)
prob_above_q75 = (df["Sales"] > q75).mean()

print(f"75th percentile of Sales: ₹{q75:,.2f}")
print(f"P(Order Sales > 75th percentile) = {prob_above_q75:.2%}")

# Demonstration of Normal, Uniform and Exponential distributions
rng = np.random.default_rng(42)
normal_sample = rng.normal(loc=df["Sales"].mean(), scale=df["Sales"].std(), size=1000)
uniform_sample = rng.uniform(df["Sales"].min(), df["Sales"].max(), size=1000)
exponential_sample = rng.exponential(scale=df["Sales"].mean(), size=1000)

fig, ax = plt.subplots(figsize=(9,5))
ax.hist(normal_sample, bins=30)
ax.set_title("Simulated Normal Distribution")
plt.tight_layout()
plt.show()

fig, ax = plt.subplots(figsize=(9,5))
ax.hist(uniform_sample, bins=30)
ax.set_title("Simulated Uniform Distribution")
plt.tight_layout()
plt.show()

fig, ax = plt.subplots(figsize=(9,5))
ax.hist(exponential_sample, bins=30)
ax.set_title("Simulated Exponential Distribution")
plt.tight_layout()
plt.show()


## 11. Decision Making Under Uncertainty

In [ ]:
# Expected value calculations from the project scenario
decision = pd.DataFrame({
    "Alternative": ["Basic Campaign", "Premium Campaign"],
    "High Demand Probability": [0.60, 0.60],
    "High Payoff": [180000, 250000],
    "Low Demand Probability": [0.40, 0.40],
    "Low Payoff": [60000, 20000]
})
decision["Expected Value"] = (
    decision["High Demand Probability"] * decision["High Payoff"]
    + decision["Low Demand Probability"] * decision["Low Payoff"]
)
display(decision)
print("Best alternative by expected value:", decision.loc[decision["Expected Value"].idxmax(), "Alternative"])


## 12. Sampling and Sampling Distribution

In [ ]:
# 1,000 random samples of n=50
rng = np.random.default_rng(42)
sample_means = np.array([
    rng.choice(df["Sales"].values, size=50, replace=False).mean()
    for _ in range(1000)
])

print(f"Mean of sample means = ₹{sample_means.mean():,.2f}")
print(f"SD of sample means = ₹{sample_means.std(ddof=1):,.2f}")
theoretical_se = df["Sales"].std(ddof=1) / np.sqrt(50)
print(f"Theoretical SE = ₹{theoretical_se:,.2f}")

fig, ax = plt.subplots(figsize=(9,5))
ax.hist(sample_means, bins=30)
ax.set_title("Sampling Distribution of Mean Sales (n=50)")
ax.set_xlabel("Sample Mean Sales (₹)")
plt.tight_layout()
plt.show()


## 13. Confidence Interval Estimation

In [ ]:
# t-based confidence intervals for the mean Sales
n = len(df)
mean_sales = df["Sales"].mean()
se = stats.sem(df["Sales"])

ci_rows = []
for level in [0.90, 0.95, 0.99]:
    alpha = 1 - level
    lower, upper = stats.t.interval(level, df=n-1, loc=mean_sales, scale=se)
    ci_rows.append([f"{level:.0%}", lower, upper])

ci_table = pd.DataFrame(ci_rows, columns=["Confidence Level", "Lower", "Upper"])
display(ci_table)


## 14. Bootstrapping

In [ ]:
# 1,000 bootstrap resamples for median sales
rng = np.random.default_rng(42)
sales_values = df["Sales"].to_numpy()

bootstrap_medians = np.array([
    np.median(rng.choice(sales_values, size=len(sales_values), replace=True))
    for _ in range(1000)
])

boot_lower, boot_upper = np.percentile(bootstrap_medians, [2.5, 97.5])
print(f"Bootstrap 95% CI for median sales: ₹{boot_lower:,.2f} to ₹{boot_upper:,.2f}")

fig, ax = plt.subplots(figsize=(9,5))
ax.hist(bootstrap_medians, bins=30)
ax.axvline(boot_lower, linestyle="--")
ax.axvline(boot_upper, linestyle="--")
ax.set_title("Bootstrap Distribution of Median Sales")
ax.set_xlabel("Median Sales (₹)")
plt.tight_layout()
plt.show()


## 15. Hypothesis Testing

In [ ]:
# H0: mean sales = ₹15,000
t_stat, p_value = stats.ttest_1samp(df["Sales"], popmean=15000)

print(f"t-statistic = {t_stat:.3f}")
print(f"p-value = {p_value:.10f}")
if p_value < 0.05:
    print("Decision: Reject H0 at α = 0.05.")
else:
    print("Decision: Fail to reject H0 at α = 0.05.")


## 16. A/B Testing (Simulated Controlled Experiment)

In [ ]:
# The project document explicitly uses simulated website experiment data.
# A: 81 conversions / 1000 visitors
# B: 92 conversions / 1000 visitors

visitors_A, conversions_A = 1000, 81
visitors_B, conversions_B = 1000, 92

rate_A = conversions_A / visitors_A
rate_B = conversions_B / visitors_B

# Two-proportion z-test
p_pool = (conversions_A + conversions_B) / (visitors_A + visitors_B)
se_pool = np.sqrt(p_pool * (1-p_pool) * (1/visitors_A + 1/visitors_B))
z = (rate_A - rate_B) / se_pool
p = 2 * (1 - stats.norm.cdf(abs(z)))

ab = pd.DataFrame({
    "Variant": ["A", "B"],
    "Visitors": [visitors_A, visitors_B],
    "Conversions": [conversions_A, conversions_B],
    "Conversion Rate": [rate_A, rate_B]
})
display(ab)

print(f"Two-proportion z = {z:.3f}")
print(f"p-value = {p:.4f}")
print("Decision:", "Statistically significant" if p < 0.05 else "Not statistically significant at 5%")


## 17. Time Series Analysis and Forecasting

In [ ]:
# Aggregate sales by month
monthly = df.set_index("Order_Date")["Sales"].resample("MS").sum()
print("Monthly observations:", len(monthly))
display(monthly.to_frame("Sales").head())

# Hold out the final four months for testing
train = monthly.iloc[:-4]
test = monthly.iloc[-4:]

# Exponential Smoothing
ets = ExponentialSmoothing(train, trend="add", seasonal=None).fit(optimized=True)
ets_forecast = ets.forecast(len(test))

# ARIMA
arima_model = ARIMA(train, order=(1,1,1)).fit()
arima_forecast = arima_model.forecast(len(test))

forecast_table = pd.DataFrame({
    "Actual": test.values,
    "Exponential Smoothing": ets_forecast.values,
    "ARIMA": arima_forecast.values
}, index=test.index)

display(forecast_table)

fig, ax = plt.subplots(figsize=(11,5))
train.plot(ax=ax, label="Train")
test.plot(ax=ax, label="Actual Test")
ets_forecast.plot(ax=ax, label="Exponential Smoothing")
arima_forecast.plot(ax=ax, label="ARIMA")
ax.set_title("Monthly Sales Forecast")
ax.set_ylabel("Sales (₹)")
ax.legend()
plt.tight_layout()
plt.show()


In [ ]:
# Forecast accuracy
def forecast_metrics(actual, predicted):
    mae = np.mean(np.abs(actual - predicted))
    rmse = np.sqrt(np.mean((actual - predicted) ** 2))
    return mae, rmse

ets_mae, ets_rmse = forecast_metrics(test.values, ets_forecast.values)
arima_mae, arima_rmse = forecast_metrics(test.values, arima_forecast.values)

accuracy = pd.DataFrame({
    "Model": ["Exponential Smoothing", "ARIMA"],
    "MAE": [ets_mae, arima_mae],
    "RMSE": [ets_rmse, arima_rmse]
})
display(accuracy)

print("Better model by MAE:", accuracy.loc[accuracy["MAE"].idxmin(), "Model"])


## 18. Optimization Modeling

In [ ]:
# Maximize profit = 1800x + 1400y
# Subject to:
# 3x + 2y <= 180
# 2x + 4y <= 160
# x, y >= 0
#
# Brute-force integer solution (avoids requiring a special optimization package).

best = None
for x in range(0, 181):
    for y in range(0, 161):
        if 3*x + 2*y <= 180 and 2*x + 4*y <= 160:
            profit = 1800*x + 1400*y
            if best is None or profit > best[0]:
                best = (profit, x, y)

max_profit, x_opt, y_opt = best
print(f"Optimal integer solution: x={x_opt}, y={y_opt}")
print(f"Maximum modeled profit = ₹{max_profit:,.2f}")
print(f"Resource 1 used = {3*x_opt + 2*y_opt} / 180")
print(f"Resource 2 used = {2*x_opt + 4*y_opt} / 160")


## 19. Simulation Modeling

In [ ]:
# Inventory safety-stock simulation.
# We simulate random demand and compare three stock multipliers.
rng = np.random.default_rng(42)
base_stock = 30
days = 10000
demand = rng.poisson(lam=30, size=days)

sim_rows = []
for multiplier in [1.0, 1.15, 1.30]:
    stock = base_stock * multiplier
    served = np.minimum(demand, stock)
    shortage = np.maximum(demand - stock, 0)
    sim_rows.append({
        "Stock Multiplier": multiplier,
        "Avg Shortage": shortage.mean(),
        "Avg Units Served": served.mean(),
        "Shortage Rate": (shortage > 0).mean()
    })

simulation_results = pd.DataFrame(sim_rows)
display(simulation_results)


## 20. ANOVA

In [ ]:
# One-way ANOVA: compare Sales across four channels
groups = [g["Sales"].values for _, g in df.groupby("Channel")]
channel_names = list(df["Channel"].dropna().unique())

f_stat, p_value = stats.f_oneway(*groups)
print("Channels:", channel_names)
print(f"F-statistic = {f_stat:.3f}")
print(f"p-value = {p_value:.6f}")
print("Decision:", "Reject H0" if p_value < 0.05 else "Fail to reject H0 at α = 0.05")


## 21. Experimental Design

### Proposed controlled experiment

- **Factor 1 — Advertising Type:** Social / Search
- **Factor 2 — Discount:** 5% / 15%
- **Response variable:** Sales
- **Experimental unit:** Comparable campaign-market cell
- **Design principle:** Randomize assignments and keep exposure comparable to reduce bias.

The project document treats this as a proposed experiment rather than historical observations.


## 22. Factorial Experiment

In [ ]:
# Simulated 2x2 factorial experiment.
# This is simulated experimental data, not historical sales data.
rng = np.random.default_rng(42)

rows = []
for advertising in ["Social", "Search"]:
    for discount in [5, 15]:
        # Effects chosen only to create an illustrative controlled experiment.
        base = 10000
        ad_effect = 1800 if advertising == "Search" else 0
        discount_effect = 1200 if discount == 15 else 0
        interaction = 300 if (advertising == "Search" and discount == 15) else 0

        for _ in range(50):
            sales = base + ad_effect + discount_effect + interaction + rng.normal(0, 1200)
            rows.append([advertising, discount, sales])

factorial_df = pd.DataFrame(rows, columns=["Advertising", "Discount", "Sales"])
display(factorial_df.groupby(["Advertising", "Discount"])["Sales"].agg(["mean", "std", "count"]))

import statsmodels.formula.api as smf
factorial_model = smf.ols("Sales ~ C(Advertising) + C(Discount) + C(Advertising):C(Discount)", data=factorial_df).fit()
anova_table = sm.stats.anova_lm(factorial_model, typ=2)
display(anova_table)

print("Interpretation: examine the p-values for the advertising main effect, discount main effect, and interaction.")


## 23. Business Insights and Findings

In [ ]:
# Automatically produce key business metrics
top_category = df.groupby("Category")["Sales"].sum().idxmax()
top_region = df.groupby("Region")["Sales"].sum().idxmax()
top_channel = df.groupby("Channel")["Sales"].sum().idxmax()

print(f"• Total modeled sales: ₹{df['Sales'].sum():,.2f}")
print(f"• Total modeled profit: ₹{df['Profit'].sum():,.2f}")
print(f"• Average order value: ₹{df['Sales'].mean():,.2f}")
print(f"• Highest-sales category: {top_category}")
print(f"• Highest-sales region: {top_region}")
print(f"• Highest-sales channel: {top_channel}")
print(f"• Return rate: {df['Returned'].mean():.2%}")
print(f"• Sales/Discount regression R²: {model.rsquared:.4f}")


## 24. Business Recommendations

1. Monitor **sales and profit together** when evaluating categories and channels.
2. Use **monthly forecasts** to support inventory and staffing decisions.
3. Target promotions toward **profitable customer, category and regional combinations**.
4. Do not assume larger discounts automatically improve outcomes; **test promotions experimentally**.
5. Use dashboard filters for **date, region, category, channel and customer segment**.
6. Set safety stock using a target service level and expected demand variability.
7. Repeat A/B testing with **real website traffic** before making permanent marketing changes.
8. Use factorial experiments to identify effective combinations of advertising and discount.


## 25. Conclusion

This notebook implements the complete Business Analytics workflow described in the supplied project document. It uses the supplied synthetic retail transaction data and keeps simulated sections—A/B testing and factorial experimentation—clearly separated from historical dataset analysis.

**Reproducibility:** random seed = 42.


### Notes
- The CSV is the source for all transaction-level calculations.
- The Word document provides the project scope, terminology and business scenario. fileciteturn0file0L43-L65
- The notebook recalculates statistics from the supplied CSV rather than hard-coding the report's displayed values.
- For Google Colab, upload `synthetic_business_sales_dataset.csv` when prompted if the local file path is unavailable.